# Atividade Avaliativa CDIA - Pipeline de Big Data

Pipeline em camadas seguindo a **Arquitetura Medallion** usando **Apache Spark**, **Silo** (S3 local) e **Delta Lake**.  

**Etapas:**
1. Inicialização do Spark com Silo (Data Lake local);
2. Validação da conexão;
3. Camada Bronze;  
  3.1. Validação e Leitura da Camada Bronze;  
  3.2. Histórico e Versionamento do Delta Lake (Time Travel);
4. Camada Silver;  
  4.1 Validação e Leitura da Camada Silver;
5. Camada Gold.

## 1. Inicialização do Spark com Suporte ao Silo

Para conectar o Spark a um serviço compatível com S3 (Container do Silo rodando na porta 9000) é necessário:
1. **JARs do Hadoop AWS e AWS Java SDK**: Fornecem as classes Java necessárias para entender a URI `s3a://` (`hadoop-aws` e `aws-java-sdk-bundle`);
2. **Endpoint local**: Aponta para `http://localhost:9000` em vez de um servidor na AWS;
3. **Credenciais**: `admin` / `password123` via `SimpleAWSCredentialsProvider`;
4. **Path-style access**: Obrigatório para MinIO/Silo (`http://endpoint/bucket` em vez de `http://bucket.endpoint`);
5. **Desativação de SSL**: O container local é HTTP puro.

In [17]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta.tables import DeltaTable

# Inicializa o SparkSession com suporte integrado a Silo e Delta Lake
spark = (
    SparkSession.builder
    .appName('Pipeline_Big_Data_CDIA')
    
    # 1. Download/Inclusão dos JARs: Hadoop AWS + AWS SDK + Delta Lake 3.2.0
    .config(
        'spark.jars.packages',
        'org.apache.hadoop:hadoop-aws:3.3.4,'
        'com.amazonaws:aws-java-sdk-bundle:1.12.262,'
        'io.delta:delta-spark_2.12:3.2.0'
    )
    
    # 2. Conexão com o Silo (porta 9000 local)
    .config('spark.hadoop.fs.s3a.endpoint', 'http://localhost:9000')
    
    # 3. Credenciais de autenticação
    .config('spark.hadoop.fs.s3a.access.key', 'admin')
    .config('spark.hadoop.fs.s3a.secret.key', 'password123')
    .config(
        'spark.hadoop.fs.s3a.aws.credentials.provider',
        'org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider'
    )
    
    # 4. Formato de acesso por caminho (Path Style) e desativação de SSL (HTTP local)
    .config('spark.hadoop.fs.s3a.path.style.access', 'true')
    .config('spark.hadoop.fs.s3a.connection.ssl.enabled', 'false')
    
    # 5. Mapeamento da classe de sistema de arquivos do S3A
    .config('spark.hadoop.fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem')
    
    # 6. Configurações do Delta Lake
    .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension')
    .config(
        'spark.sql.catalog.spark_catalog',
        'org.apache.spark.sql.delta.catalog.DeltaCatalog'
    )
    
    .getOrCreate()
)

print('✓ SparkSession conectada ao Silo e Delta Lake com sucesso!')
print(f'✓ Aplicação: {spark.sparkContext.appName}')
print(f'✓ Spark Web UI: http://localhost:4040')
print(f'✓ Silo Console UI: http://localhost:9001')

✓ SparkSession conectada ao Silo e Delta Lake com sucesso!
✓ Aplicação: Pipeline_Big_Data_CDIA
✓ Spark Web UI: http://localhost:4040
✓ Silo Console UI: http://localhost:9001


## 2. Teste de Conexão
#### Leitura da Camada Landing

Lemos os dados brutos gerados pelo `gerador-dados.ipynb` no caminho `s3a://lakehouse/landing/vendas_cruas_csv`. 
Na Landing, o dado ainda está em CSV com separador `;` e todos os tipos de dados são tratados como texto (`string`).

In [18]:
CAMINHO_LANDING = 's3a://lakehouse/landing/vendas_cruas_csv'

df_landing = (
    spark.read
    .format('csv')
    .option('header', 'true')
    .option('sep', ';')
    .load(CAMINHO_LANDING)
)

print(f'Total de registros na Landing Zone: {df_landing.count():,}')
df_landing.printSchema()
df_landing.show(5, truncate=False)

Total de registros na Landing Zone: 1,010,000
root
 |-- id_venda: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- filial_uf: string (nullable = true)
 |-- forma_pagamento: string (nullable = true)
 |-- quantidade: string (nullable = true)
 |-- preco_unitario: string (nullable = true)
 |-- desconto: string (nullable = true)

+--------+-----------+---------+---------------+----------+--------------+--------+
|id_venda|categoria  |filial_uf|forma_pagamento|quantidade|preco_unitario|desconto|
+--------+-----------+---------+---------------+----------+--------------+--------+
|875000  |NULL       |RS       |Boleto         |1         |189,76        |0.0     |
|875001  |livros     |SC       |PIX            |2         |-8.47         |0.05    |
|875002  |INFORMÁTICA|PR       |Cartão         |3         |363,41        |0.0     |
|875003  |Casa       |SP       |Boleto         |4         |245.49        |0.0     |
|875004  |Esporte    |RS       |PIX            |5         |62,7

## 3. Camada Bronze

Nenhuma limpeza, filtro ou transformação foi realizada para a camada bronze, o dado cru foi preservado da forma como chegou, apenas as colunas de metadados de auditoria `_data_ingestao` (timestamp do momento da ingestão do registro) e `_arquivo_origem` (nome do arquivo de origem) foram adicionadas, por fim foi realizada a gravação em formato Delta.

In [19]:
# Adição dos metadados de auditoria
df_bronze = (
    df_landing
    .withColumn('_data_ingestao', F.current_timestamp())
    .withColumn('_arquivo_origem', F.input_file_name())
)

# Gravação da tabela Delta na camada Bronze
CAMINHO_BRONZE = 's3a://lakehouse/bronze/vendas'

(
    df_bronze.write
    .format('delta')
    .mode('overwrite')
    .save(CAMINHO_BRONZE)
)

print('✓ Dados gravados com sucesso na Camada Bronze.')

26/11/23 14:25:15 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


✓ Dados gravados com sucesso na Camada Bronze.


### 3.1 Validação e Leitura da Camada Bronze

Lemos os dados gravados a partir do caminho Delta para conferir o schema e os metadados gerados.

In [20]:
df_bronze_read = spark.read.format('delta').load(CAMINHO_BRONZE)

print(f'Total de registros na Camada Bronze: {df_bronze_read.count():,}')
df_bronze_read.printSchema()
df_bronze_read.show(5, truncate=False)

Total de registros na Camada Bronze: 1,010,000
root
 |-- id_venda: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- filial_uf: string (nullable = true)
 |-- forma_pagamento: string (nullable = true)
 |-- quantidade: string (nullable = true)
 |-- preco_unitario: string (nullable = true)
 |-- desconto: string (nullable = true)
 |-- _data_ingestao: timestamp (nullable = true)
 |-- _arquivo_origem: string (nullable = true)

+--------+-----------+---------+---------------+----------+--------------+--------+--------------------------+-------------------------------------------------------------------------------------------------+
|id_venda|categoria  |filial_uf|forma_pagamento|quantidade|preco_unitario|desconto|_data_ingestao            |_arquivo_origem                                                                                  |
+--------+-----------+---------+---------------+----------+--------------+--------+--------------------------+--------------------------

### 3.2 Histórico e Versionamento do Delta Lake (Time Travel)

O Delta Lake armazena um log de transações (`_delta_log`). É inspecionar o histórico com `DeltaTable.forPath()` para auditar todas as operações realizadas.  
Exemplo apenas para a camada bronze.

In [21]:
tabela_delta_bronze = DeltaTable.forPath(spark, CAMINHO_BRONZE)
tabela_delta_bronze.history().select('version', 'timestamp', 'operation', 'operationMetrics').show(truncate=False)

+-------+-------------------+---------+---------------------------------------------------------------------+
|version|timestamp          |operation|operationMetrics                                                     |
+-------+-------------------+---------+---------------------------------------------------------------------+
|2      |2026-11-23 14:25:18|WRITE    |{numFiles -> 8, numOutputRows -> 1010000, numOutputBytes -> 10057389}|
|1      |2026-11-23 14:24:18|WRITE    |{numFiles -> 8, numOutputRows -> 1010000, numOutputBytes -> 10057389}|
|0      |2026-11-23 11:22:05|WRITE    |{numFiles -> 8, numOutputRows -> 1010000, numOutputBytes -> 10057389}|
+-------+-------------------+---------+---------------------------------------------------------------------+



## 4. Camada Silver

Nesta etapa, os seguintes passos são realizados:
- Leitura e carregamento dos dados da camada bronze;
- Eliminação dos registros duplicados;
- Tratamento de valores decimais mal formatados;
- Conversão de tipagens;
- Tratamento de colunas string;
- Tratamento de valores nulos;
- Tratamento de anomalias de preços;
- Enriquecimento e engenharia de atributos;
- Gravação final da camada silver.

In [22]:
# Leitura dos dados da camada bronze
CAMINHO_BRONZE = 's3a://lakehouse/bronze/vendas'
CAMINHO_SILVER = 's3a://lakehouse/silver/vendas'

# Carregamento da camada bronze
df_bronze_source = spark.read.format('delta').load(CAMINHO_BRONZE)

# Eliminação de registros duplicados
df_silver_dedup = df_bronze_source.dropDuplicates(['id_venda'])

# Correção de valores e tipagens
df_silver_typed = (
    df_silver_dedup
    .withColumn('id_venda', F.col('id_venda').cast('long'))
    .withColumn('quantidade', F.col('quantidade').cast('integer'))
    .withColumn(
        'preco_unitario',
        F.regexp_replace(F.col('preco_unitario'), ',', '.').cast('decimal(10, 2)')
    )
    .withColumn('desconto', F.col('desconto').cast('decimal(10, 2)'))
    .withColumn('filial_uf', F.trim(F.col('filial_uf')))
    .withColumn('forma_pagamento', F.trim(F.col('forma_pagamento')))
)

# Padronização de valores string e tratamento de nulos
categoria_padronizada = (
    F.when(
        F.col('categoria').isNull() | (F.trim(F.col('categoria')) == '') | (F.upper(F.trim(F.col('categoria'))) == 'NULL'),
        F.lit('Não Informado')
    )
    .when(F.lower(F.trim(F.col('categoria'))).isin('eletrônicos', 'eletronicos'), F.lit('Eletrônicos'))
    .when(F.lower(F.trim(F.col('categoria'))).isin('informática', 'informatica'), F.lit('Informática'))
    .when(F.lower(F.trim(F.col('categoria'))) == 'livros', F.lit('Livros'))
    .when(F.lower(F.trim(F.col('categoria'))) == 'casa', F.lit('Casa'))
    .when(F.lower(F.trim(F.col('categoria'))) == 'esporte', F.lit('Esporte'))
    .otherwise(F.initcap(F.trim(F.col('categoria'))))
)

# Filtragem de anomalias (preços negativos e zerados)
df_silver_tratado = (
    df_silver_typed
    .withColumn('categoria', categoria_padronizada)
    .filter((F.col('preco_unitario') > 0) & (F.col('quantidade') > 0))
)

# Camada Silver pós tratamento
print('✓ 10 primeiras linhas tratadas:')
df_silver_tratado.show(10, truncate=False)

✓ 10 primeiras linhas tratadas:


+--------+-----------+---------+---------------+----------+--------------+--------+--------------------------+-------------------------------------------------------------------------------------------------+
|id_venda|categoria  |filial_uf|forma_pagamento|quantidade|preco_unitario|desconto|_data_ingestao            |_arquivo_origem                                                                                  |
+--------+-----------+---------+---------------+----------+--------------+--------+--------------------------+-------------------------------------------------------------------------------------------------+
|100010  |Livros     |PR       |Boleto         |1         |126.85        |0.00    |2026-11-23 14:25:15.405638|s3a://lakehouse/landing/vendas_cruas_csv/part-00000-7e70ed19-3628-4e9f-99a8-2580272b07d8-c000.csv|
|100014  |Eletrônicos|PR       |PIX            |5         |76.19         |0.05    |2026-11-23 14:25:15.405638|s3a://lakehouse/landing/vendas_cruas_csv/part-00000-7e

In [23]:
# Engenharia de Atributos:
# - valor_bruto: faturamento bruto (quantidade * preco_unitario)
# - valor_desconto: montante do desconto (valor_bruto * desconto)
# - valor_liquido: faturamento líquido (valor_bruto - valor_desconto)
# - regiao: mapeamento de filial_uf para região geográfica (Sul / Sudeste)
df_silver_final = (
    df_silver_tratado
    .withColumn('valor_bruto', F.round((F.col('quantidade') * F.col('preco_unitario')), 2).cast('decimal(10, 2)'))
    .withColumn('valor_desconto', F.round((F.col('valor_bruto') * F.col('desconto')), 2).cast('decimal(10, 2)'))
    .withColumn('valor_liquido', F.round((F.col('valor_bruto') - F.col('valor_desconto')), 2).cast('decimal(10, 2)'))
    .withColumn(
        'regiao',
        F.when(F.col('filial_uf').isin('RS', 'SC', 'PR'), F.lit('Sul'))
         .when(F.col('filial_uf') == 'SP', F.lit('Sudeste'))
         .otherwise(F.lit('Outra'))
    )
    .withColumn('_data_processamento_silver', F.current_timestamp())
)

# Gravação do resultado enriquecido na tabela delta silver
(
    df_silver_final.write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .save(CAMINHO_SILVER)
)

print('✓ Camada Silver final:')
df_silver_final.show(10, truncate=False)

26/11/23 14:25:24 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


✓ Camada Silver final:


+--------+-----------+---------+---------------+----------+--------------+--------+--------------------------+-------------------------------------------------------------------------------------------------+-----------+--------------+-------------+------+--------------------------+
|id_venda|categoria  |filial_uf|forma_pagamento|quantidade|preco_unitario|desconto|_data_ingestao            |_arquivo_origem                                                                                  |valor_bruto|valor_desconto|valor_liquido|regiao|_data_processamento_silver|
+--------+-----------+---------+---------------+----------+--------------+--------+--------------------------+-------------------------------------------------------------------------------------------------+-----------+--------------+-------------+------+--------------------------+
|100010  |Livros     |PR       |Boleto         |1         |126.85        |0.00    |2026-11-23 14:25:15.405638|s3a://lakehouse/landing/vendas_cruas_c

### 4.1 Validação e Leitura da Camada Silver

Lemos a tabela Delta Silver para validar a contagem de registros resultantes, a tipagem estrita do schema e a padronização dos dados.

In [24]:
# Leitura e validação da camada silver
df_silver_read = spark.read.format('delta').load(CAMINHO_SILVER)

print(f'Total de registros válidos na Camada Silver: {df_silver_read.count():,}')
df_silver_read.printSchema()
df_silver_read.show(10, truncate=False)

print('Distribuição por Categoria Padronizada:')
df_silver_read.groupBy('categoria').count().orderBy('categoria').show()

Total de registros válidos na Camada Silver: 872,561
root
 |-- id_venda: long (nullable = true)
 |-- categoria: string (nullable = true)
 |-- filial_uf: string (nullable = true)
 |-- forma_pagamento: string (nullable = true)
 |-- quantidade: integer (nullable = true)
 |-- preco_unitario: decimal(10,2) (nullable = true)
 |-- desconto: decimal(10,2) (nullable = true)
 |-- _data_ingestao: timestamp (nullable = true)
 |-- _arquivo_origem: string (nullable = true)
 |-- valor_bruto: decimal(10,2) (nullable = true)
 |-- valor_desconto: decimal(10,2) (nullable = true)
 |-- valor_liquido: decimal(10,2) (nullable = true)
 |-- regiao: string (nullable = true)
 |-- _data_processamento_silver: timestamp (nullable = true)

+--------+-----------+---------+---------------+----------+--------------+--------+--------------------------+-------------------------------------------------------------------------------------------------+-----------+--------------+-------------+------+-------------------------

## 5. Camada Gold

Abordagens realizadas:
1. **5.1 Tabela Analítica Única:** Tabela analítica agregada com as principais métricas de vendas e receita, permitindo fatiar por região, filial, categoria e forma de pagamento;
2. **5.2 Modelo Dimensional (Star Schema):** Modelagem clássica com Tabela Fato (`fact_vendas`) e Dimensões (`dim_filial`, `dim_categoria`, `dim_pagamento`).

### 5.1 Tabela Analítica Única

Nesta abordagem, agrupamos os dados por `regiao`, `filial_uf`, `categoria` e `forma_pagamento`, pré-calculando os principais indicadores chaves de desempenho (KPI) do negócio:
- `total_pedidos`: volume total de transações;
- `total_itens_vendidos`: soma das quantidades vendidas;
- `faturamento_bruto`: receita bruta total antes dos descontos;
- `total_descontos`: montante total de descontos concedidos;
- `faturamento_liquido`: receita líquida efetiva recebida;
- `ticket_medio_pedido`: valor médio gasto por transação.

Essa tabela reduz 872 mil registros a uma visão compacta para relatórios.

In [ ]:
# Leitura dos dados da camada silver
CAMINHO_SILVER = 's3a://lakehouse/silver/vendas'
CAMINHO_GOLD = 's3a://lakehouse/gold/vendas_consolidadas'

df_silver = spark.read.format('delta').load(CAMINHO_SILVER)

# Criação da tabela
df_unified = (
    df_silver
    .groupBy('regiao', 'filial_uf', 'categoria', 'forma_pagamento')
    .agg(
        F.count('id_venda').alias('total_pedidos'),
        F.sum('quantidade').alias('total_itens_vendidos'),
        F.round(F.sum('valor_bruto'), 2).alias('faturamento_bruto'),
        F.round(F.sum('valor_desconto'), 2).alias('total_descontos'),
        F.round(F.sum('valor_liquido'), 2).alias('faturamento_liquido'),
        F.round(F.avg('valor_liquido'), 2).alias('ticket_medio_pedido')
    )
    .orderBy('regiao', 'filial_uf', 'faturamento_liquido', ascending=[True, True, False])
)

# Gravação da tabela Delta na camada Gold
(
    df_unified.write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .save(CAMINHO_GOLD)
)

print(f'✓ Tabela analítica gravada com sucesso! Total de agregações: {df_unified.count()} linhas')
df_unified.show(10, truncate=False)

✓ Cubo Analítico Gold gravado com sucesso! Total de agregações: 66 linhas
+-------+---------+-----------+---------------+-------------+--------------------+-----------------+---------------+-------------------+-------------------+
|regiao |filial_uf|categoria  |forma_pagamento|total_pedidos|total_itens_vendidos|faturamento_bruto|total_descontos|faturamento_liquido|ticket_medio_pedido|
+-------+---------+-----------+---------------+-------------+--------------------+-----------------+---------------+-------------------+-------------------+
|Sudeste|SP       |Casa       |Cartão         |21457        |128856              |29226780.68      |0.00           |29226780.68        |1362.11            |
|Sudeste|SP       |Eletrônicos|Boleto         |21448        |128504              |29099671.98      |0.00           |29099671.98        |1356.75            |
|Sudeste|SP       |Casa       |Boleto         |21504        |128692              |28941146.00      |0.00           |28941146.00        |1345.

### 5.2 Modelo Dimensional Star Schema (Tabela Fato e Dimensões)

Nesta abordagem, as seguintes foram implementadas:
- **`dim_filial`:** Dimensão geográfica (`filial_uf`, `regiao`);
- **`dim_categoria`:** Dimensão de produtos (`categoria`);
- **`dim_pagamento`:** Dimensão de meios de pagamento (`forma_pagamento`, `desconto_padrao`);
- **`fact_vendas`:** Tabela fato granular contendo as métricas numéricas e as chaves de relacionamento com as dimensões.

In [10]:
# 1. Modelagem das Dimensões
dim_filial = df_silver.select('filial_uf', 'regiao').distinct().orderBy('filial_uf')
dim_categoria = df_silver.select('categoria').distinct().orderBy('categoria')
dim_pagamento = (
    df_silver
    .select('forma_pagamento')
    .distinct()
    .withColumn('desconto_padrao', F.when(F.col('forma_pagamento') == 'PIX', F.lit(0.05)).otherwise(F.lit(0.0)))
    .orderBy('forma_pagamento')
)

# 2. Modelagem da Tabela Fato
fact_vendas = df_silver.select(
    'id_venda',
    'filial_uf',
    'categoria',
    'forma_pagamento',
    'quantidade',
    'preco_unitario',
    'valor_bruto',
    'valor_desconto',
    'valor_liquido'
)

# 3. Gravação das tabelas Delta do Star Schema na Gold
dim_filial.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').save('s3a://lakehouse/gold/dim_filial')
dim_categoria.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').save('s3a://lakehouse/gold/dim_categoria')
dim_pagamento.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').save('s3a://lakehouse/gold/dim_pagamento')
fact_vendas.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').save('s3a://lakehouse/gold/fact_vendas')

print('✓ Tabelas do Star Schema gravadas na Gold com sucesso!')
print('Dimensão Filial:')
dim_filial.show(truncate=False)
print('Dimensão Pagamento:')
dim_pagamento.show(truncate=False)
print('Amostra da Tabela Fato:')
fact_vendas.show(5, truncate=False)

26/11/26 14:34:03 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


✓ Tabelas do Star Schema gravadas na Gold com sucesso!
Dimensão Filial:
+---------+-------+
|filial_uf|regiao |
+---------+-------+
|PR       |Sul    |
|RS       |Sul    |
|SC       |Sul    |
|SP       |Sudeste|
+---------+-------+

Dimensão Pagamento:
+---------------+---------------+
|forma_pagamento|desconto_padrao|
+---------------+---------------+
|Boleto         |0.0            |
|Cartão         |0.0            |
|PIX            |0.05           |
+---------------+---------------+

Amostra da Tabela Fato:
+--------+---------+---------+---------------+----------+--------------+-----------+--------------+-------------+
|id_venda|filial_uf|categoria|forma_pagamento|quantidade|preco_unitario|valor_bruto|valor_desconto|valor_liquido|
+--------+---------+---------+---------------+----------+--------------+-----------+--------------+-------------+
|1       |SC       |Livros   |Cartão         |2         |204.80        |409.60     |0.00          |409.60       |
|100012  |RS       |Casa    